### Сбор основной витрины

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('data/MD_base.csv')
df

/var/folders/8d/cnlnd7c93fj81tn1mzz4c1nm0000gn/T/ipykernel_2345/696777288.py:1: DtypeWarning: Columns (4,5,6,13,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/MD_base.csv')


,Отчетный период,Страна-партнер,Направление,Код товара (2 знака),Код товара (4 знака),Код товара (6 знаков),Код товара (8 знаков),Код товара (10 знаков),Значение (стоимость),Единицы стоимости,Значение (масса),Единица объема,Дополнительная единица измерения (ДЭИ),"ДЭИ, описание",Значение (стоимость) - ДЭИ,Единица стоимости
0,2025-08-01,CША,Экспорт,1.0,106,10619,1061991.0,106199195.0,23520.000000,USD,382.0,NO,0.0,NaN,NaN,NaN
1,2025-08-01,CША,Экспорт,8.0,813,81340,8134020.0,813402060.0,3250.000000,USD,512.0,KG,0.0,NaN,NaN,NaN
2,2025-08-01,CША,Экспорт,9.0,901,90121,9012100.0,901210049.0,4261.000000,USD,412.0,KG,0.0,NaN,NaN,NaN
3,2025-08-01,CША,Экспорт,9.0,902,90210,9021010.0,902101050.0,8199.000000,USD,323.0,KG,0.0,NaN,NaN,NaN
4,2025-08-01,CША,Экспорт,9.0,902,90230,9023000.0,902300090.0,9679.000000,USD,358.0,KG,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4603442,2019-01-01,Ukraine,Импорт,96.0,9608,960860,NaN,NaN,700.951650,USD,30.0,килограмм,NaN,NaN,NaN,NaN
4603443,2019-01-01,Ukraine,Импорт,96.0,9609,960910,NaN,NaN,6268.608326,USD,694.0,килограмм,NaN,NaN,NaN,NaN
4603444,2019-01-01,Ukraine,Импорт,96.0,9613,961380,NaN,NaN,38.814912,USD,1.0,килограмм,NaN,NaN,NaN,NaN
4603445,2019-01-01,Ukraine,Импорт,96.0,9617,961700,NaN,NaN,1070.834932,USD,72.0,килограмм,NaN,NaN,NaN,NaN


In [3]:
df['Отчетный период'] = pd.to_datetime(df['Отчетный период'])
df['Код товара (6 знаков)'] = df['Код товара (6 знаков)'].astype(str).str.replace('.0', '').str.zfill(6)
df['Код товара (4 знака)'] = df['Код товара (4 знака)'].astype(str).str.replace('.0', '').str.zfill(4)
df['Код товара (2 знака)'] = df['Код товара (2 знака)'].astype(str).str.replace('.0', '').str.zfill(2)
df['Значение (стоимость)'] = df['Значение (стоимость)'].replace({np.nan: 0, 'nan': 0}).astype(float)
df['Значение (масса)'] = df['Значение (масса)'].replace({np.nan: 0, 'nan': 0}).astype(float)
df['Дополнительная единица измерения (ДЭИ)'] = df['Дополнительная единица измерения (ДЭИ)'].replace({np.nan: 0, 'nan': 0}).astype(float)

df['Значение (стоимость) - ДЭИ'] = df['Значение (стоимость) - ДЭИ'].replace({np.nan: 0, 'nan': 0}).astype(float)
df.loc[df['Единица объема'] == 'тонн', 'Значение (масса)'] = df.loc[df['Единица объема'] == 'тонн', 'Значение (масса)'] * 1000
df['Единица объема'] = df['Единица объема'].replace({'килограмм': 'KG', 'Kilogram': 'KG',
        'KG': 'KG', 'KILOGRAM': 'KG', 'KGS': 'KG', 'тонн': 'KG', 'ＫＧ': 'KG'})
df['Единица объема'] = df['Единица объема'].replace({'NO': 'UNIT',
        'UNIT': 'UNIT', 'NOS': 'UNIT', 'Number of item': 'UNIT', 'ＮＯ': 'UNIT'})

In [4]:
# ТЕХНИЧЕСКАЯ ЗАМЕНА

thai_901839 = pd.read_csv('data/thai_9018_massa.csv')
thai_901839['Отчетный период'] = pd.to_datetime(thai_901839['Отчетный период'])
thai_901839['Страна-партнер'] = 'Таиланд'
thai_901839['Код товара (6 знаков)'] = '901839'
thai_901839 = thai_901839.rename(columns={'Значение (масса)': 'mass'})

df = df.merge(thai_901839, on=['Отчетный период', 'Страна-партнер', 'Код товара (6 знаков)'], how='left')
df.loc[(df['Страна-партнер'] == 'Таиланд') &
       (df['Код товара (6 знаков)'] == '901839'), 'Значение (масса)'] = df.loc[(df['Страна-партнер'] == 'Таиланд') &
       (df['Код товара (6 знаков)'] == '901839'), 'mass']
df.loc[(df['Страна-партнер'] == 'Таиланд') &
       (df['Код товара (6 знаков)'] == '901839'), 'Единица объема'] = 'KG'
df = df.drop(columns='mass')

In [5]:
# ТЕХНИЧЕСКАЯ ЗАМЕНА

thai_901839 = pd.read_csv('data/thai_90189090_massa.csv')
thai_901839['Отчетный период'] = pd.to_datetime(thai_901839['Отчетный период'])
thai_901839['Страна-партнер'] = 'Таиланд'
thai_901839['Код товара (6 знаков)'] = '901890'
thai_901839 = thai_901839.rename(columns={'Значение (масса)': 'mass'})

df = df.merge(thai_901839, on=['Отчетный период', 'Страна-партнер', 'Код товара (6 знаков)'], how='left')
df.loc[(df['Страна-партнер'] == 'Таиланд') &
       (df['Код товара (6 знаков)'] == '901890'), 'Значение (масса)'] = df.loc[(df['Страна-партнер'] == 'Таиланд') &
       (df['Код товара (6 знаков)'] == '901890'), 'mass']
df.loc[(df['Страна-партнер'] == 'Таиланд') &
       (df['Код товара (6 знаков)'] == '901890'), 'Единица объема'] = 'UNIT'
df = df.drop(columns='mass')

In [6]:
# ТЕХНИЧЕСКАЯ ПОПРАВКА

mask = (df['Страна-партнер'] == 'Республика Корея') & (df['Отчетный период'].dt.year == 2025)

df.loc[mask, 'Направление'] = df.loc[mask, 'Направление'].replace({
    'Экспорт': 'Импорт',
    'Импорт': 'Экспорт'
})

In [7]:
# ТЕХНИЧЕСКАЯ ПОПРАВКА

mask = (df['Страна-партнер'] == 'Бразилия') & (df['Отчетный период'].dt.year == 2025)

df.loc[mask, 'Направление'] = df.loc[mask, 'Направление'].replace({
    'Экспорт': 'Импорт',
    'Импорт': 'Экспорт'
})

In [8]:
# ТЕХНИЧЕСКАЯ ПОПРАВКА

mask = (df['Страна-партнер'] == 'Тайвань') & (df['Отчетный период'].dt.year == 2025)

df.loc[mask, 'Направление'] = df.loc[mask, 'Направление'].replace({
    'Экспорт': 'Импорт',
    'Импорт': 'Экспорт'
})

In [9]:
# df = df.groupby(
#     ['Отчетный период', 'Страна-партнер', 'Направление',
#      'Код товара (2 знака)', 'Код товара (4 знака)', 'Единицы стоимости',
#      'Единица объема'],
#     as_index=False,
#     dropna=False
# )[['Значение (стоимость)',
#    'Значение (стоимость) - ДЭИ',
#    'Значение (масса)']].sum()
# df = df[df['Код товара (4 знака)'].isin(['9018', '9019', '9020', '9021', '9022', '9025', '9027', '9030', '9031'])]
# df = df[df['Направление'] == 'Импорт']
# df

# БЕЗ МАССЫ
df = df.groupby(
    ['Отчетный период', 'Страна-партнер', 'Направление',
     'Код товара (2 знака)', 'Код товара (4 знака)', 'Единицы стоимости'],
    as_index=False,
    dropna=False
)[['Значение (стоимость)',
   'Значение (стоимость) - ДЭИ']].sum()
df = df[df['Код товара (4 знака)'].isin(['9018', '9019', '9020', '9021', '9022', '9025', '9027', '9030', '9031'])]
df = df[df['Направление'] == 'Импорт']
df

,Отчетный период,Страна-партнер,Направление,Код товара (2 знака),Код товара (4 знака),Единицы стоимости,Значение (стоимость),Значение (стоимость) - ДЭИ
537,2019-01-01,Austria,Импорт,90,9018,USD,3.419788e+05,0.0
538,2019-01-01,Austria,Импорт,90,9021,USD,2.525504e+05,0.0
539,2019-01-01,Austria,Импорт,90,9022,USD,2.842393e+04,0.0
542,2019-01-01,Austria,Импорт,90,9025,USD,3.857517e+03,0.0
544,2019-01-01,Austria,Импорт,90,9027,USD,1.023569e+06,0.0
...,...,...,...,...,...,...,...,...
2090726,2025-10-01,Тайвань,Импорт,90,9019,USD,7.100000e+04,0.0
2090727,2025-10-01,Тайвань,Импорт,90,9020,USD,6.500000e+04,0.0
2090728,2025-10-01,Тайвань,Импорт,90,9021,USD,1.367000e+06,0.0
2090730,2025-10-01,Тайвань,Импорт,90,9027,USD,1.670000e+05,0.0


Фильтры на дату. Удалим страны, которые берем из Comtrade. Переименуем страны

In [10]:
df = df[(df['Отчетный период'] >= '2019-01-01') & (df['Отчетный период'] <= '2025-06-01')]
df = df[~df['Страна-партнер'].isin(['Азербайджан', 'Киргизия', 'Moldova, Republic of', 'United Kingdom'])]

In [11]:
countries_map = {
        "Belgium (incl. Luxembourg 'LU' -> 1998)": 'Belgium', 'CША': 'USA',
        "France (incl. Saint Barthélemy 'BL' -> 2012; incl. French Guiana 'GF', Guadeloupe 'GP', Martinique 'MQ', Réunion 'RE' from 1997; incl. Mayotte 'YT' from 2014)": 'France',
        "Germany (incl. German Democratic Republic 'DD' from 1991)": 'Germany',
        'Ireland (Eire)': 'Ireland', "Italy (incl. San Marino 'SM' -> 1993)": 'Italy',
       "Norway (incl. Svalbard and Jan Mayen 'SJ' -> 1994 and again from 1997)": 'Norway',
       "Spain (incl. Canary Islands 'XB' from 1997)": 'Spain',
       "Switzerland (incl. Liechtenstein 'LI' -> 1994)": 'Switzerland', 'Türkiye': 'Turkey',
       'Бразилия': 'Brazil', 'Гонконг': 'Hong Kong', 'Индия': 'India', 'Китай': 'China',
       'Республика Корея': 'South Korea', 'Таджикистан': 'Tajikistan',
       'Таиланд': 'Thailand', 'Тайвань': 'Taiwan', 'Эквадор': 'Ecuador', 'Казахстан': 'Kazakhstan',
       'Япония': 'Japan', 'United Kingdom (Northern Ireland)': 'Northern Ireland', 'Moldova, Republic of': 'Moldova'
}

df['Страна-партнер'] = df['Страна-партнер'].map(countries_map).fillna(df['Страна-партнер'])

In [12]:
duplicates = df[
    df.duplicated(
        subset=['Отчетный период', 'Страна-партнер', 'Код товара (4 знака)'],
        keep=False
    )
]

duplicates

,Отчетный период,Страна-партнер,Направление,Код товара (2 знака),Код товара (4 знака),Единицы стоимости,Значение (стоимость),Значение (стоимость) - ДЭИ


Отформатируем колонки

In [13]:
df = df.rename(columns={'Отчетный период': 'rep_date', 'Страна-партнер': 'country',
       'Код товара (4 знака)': 'hs', 'Значение (стоимость)': 'value'})
df = df[['rep_date', 'country', 'hs', 'value']]
df

,rep_date,country,hs,value
537,2019-01-01,Austria,9018,3.419788e+05
538,2019-01-01,Austria,9021,2.525504e+05
539,2019-01-01,Austria,9022,2.842393e+04
542,2019-01-01,Austria,9025,3.857517e+03
544,2019-01-01,Austria,9027,1.023569e+06
...,...,...,...,...
2042148,2025-06-01,Japan,9018,3.047172e+06
2042149,2025-06-01,Japan,9019,4.138370e+04
2042150,2025-06-01,Japan,9021,1.927689e+05
2042151,2025-06-01,Japan,9022,1.352647e+05


In [14]:
df.to_excel('data/main_data.xlsx', index=False)